# Zomato vs. Swiggy: Two Bets, One Industry**Author:** [Your Name]  **Last updated:** May 2026  **Background:** Transitioning from QA to data analytics. This is a self-directed analytics project comparing India's two listed food/delivery operators across both their B2C business lines: food delivery and quick commerce.## The starting question> *Everyone says "Zomato vs Swiggy" as if they're still in the same race. Are they? What does the public data actually show?*## Approach8 quarters of public quarterly disclosures (Q1 FY25 → Q4 FY26) for both companies, both business lines. The analysis goes:1. **Layer 1 — Food delivery comparison.** Zomato vs Swiggy: GOV, growth, EBITDA margin.2. **Layer 2 — Quick commerce comparison.** Blinkit vs Instamart: GOV, contribution margin, dark store productivity, AOV.3. **Synthesis — Business mix.** How does each parent allocate B2C GOV? When did the strategies diverge?## My QA-to-analytics angleTwo QA muscles I'm leaning on:- **Edge-case skepticism** — any number that looks too clean usually isn't- **Failure-mode thinking** — what would have to be true for this trend to reverse?I flag both throughout.

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltimport numpy as npplt.rcParams['figure.figsize'] = (11, 5)plt.rcParams['axes.spines.top'] = Falseplt.rcParams['axes.spines.right'] = Falseplt.rcParams['font.family'] = 'DejaVu Sans'ZOM = '#ef4444'   # Zomato redSWG = '#f59e0b'   # Swiggy orangefd = pd.read_csv('../data/food_delivery_quarterly.csv', parse_dates=['quarter_end_date'])qc = pd.read_csv('../data/quick_commerce_quarterly.csv', parse_dates=['quarter_end_date'])fd = fd.sort_values(['company', 'quarter_end_date']).reset_index(drop=True)qc = qc.sort_values(['company', 'quarter_end_date']).reset_index(drop=True)print(f'Food delivery: {len(fd)} rows, {fd["quarter"].nunique()} quarters')print(f'Quick commerce: {len(qc)} rows, {qc["quarter"].nunique()} quarters')

---## Layer 1: Food delivery comparison (Zomato vs Swiggy)This is the recognizable race — the two apps every Indian has on their phone.

In [ ]:
# Q4 FY26 state of playfd_latest = fd[fd['quarter']=='Q4FY26'][['company','gov_inr_cr','revenue_inr_cr','adj_ebitda_inr_cr','adj_ebitda_margin_pct_of_gov','mtu_millions','aov_inr']]fd_latest.set_index('company').T

In [ ]:
# Chart 1: Food delivery GOV trajectoryfig, ax = plt.subplots()for company, color, label in [('Eternal', ZOM, 'Zomato'), ('Swiggy', SWG, 'Swiggy')]:    d = fd[fd['company']==company]    ax.plot(d['quarter_end_date'], d['gov_inr_cr'], marker='o', linewidth=2.5, color=color, label=label)    ax.fill_between(d['quarter_end_date'], 0, d['gov_inr_cr'], color=color, alpha=0.08)ax.set_title('Food delivery GOV by quarter (₹ crore)', fontsize=13, weight='bold', loc='left')ax.set_ylabel('GOV (₹ cr)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='upper left')plt.tight_layout()plt.show()

**Read:** Zomato leads on absolute GOV by ~33% — and that gap has been stable for six quarters. Both companies grow at the same pace. The race in absolute terms is steady-state.

In [ ]:
# YoY growth rate comparisonfd['gov_yoy_pct'] = fd.groupby('company')['gov_inr_cr'].pct_change(4) * 100growth_pivot = fd.pivot(index='quarter', columns='company', values='gov_yoy_pct').round(1)growth_pivot = growth_pivot.reindex(['Q1FY26','Q2FY26','Q3FY26','Q4FY26'])print('YoY GOV growth (%) — FY26 quarters only:')growth_pivot

In [ ]:
# Chart 2: EBITDA margin convergence — Layer 1 headlinefig, ax = plt.subplots()for company, color, label in [('Eternal', ZOM, 'Zomato'), ('Swiggy', SWG, 'Swiggy')]:    d = fd[fd['company']==company]    ax.plot(d['quarter_end_date'], d['adj_ebitda_margin_pct_of_gov'], marker='o', linewidth=2.5, color=color, label=label)ax.set_title('Food delivery — Adj. EBITDA margin (% of GOV)', fontsize=13, weight='bold', loc='left')ax.set_ylabel('EBITDA margin (%)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='lower right')plt.tight_layout()plt.show()

In [ ]:
# The convergence tablemargin_pivot = fd.pivot(index='quarter_end_date', columns='company', values='adj_ebitda_margin_pct_of_gov')margin_pivot['Gap (pp)'] = (margin_pivot['Eternal'] - margin_pivot['Swiggy']).round(2)margin_pivot = margin_pivot.rename(columns={'Eternal': 'Zomato', 'Swiggy': 'Swiggy'})margin_pivot

**Layer 1 finding:**The food delivery EBITDA margin gap collapsed from **2.3pp (Q1 FY25) to 0.1pp (Q4 FY26)**. Swiggy caught up entirely. Both companies now earn ~3.3–3.4% EBITDA on every rupee of GOV.This is genuinely surprising. The popular 2022–23 narrative framed Zomato as the smarter operator. By Q4 FY26, the data says: not anymore. **Food delivery is now a converged two-player commodity in India.**If both businesses grow at the same rate and earn the same margin, neither company can win the future by being better at food delivery. The race has moved.

---## Layer 2: Quick commerce comparison (Blinkit vs Instamart)The race that's actually still being run.

In [ ]:
# Q4 FY26 state of playqc_latest = qc[qc['quarter']=='Q4FY26'][['company','gov_inr_cr','revenue_inr_cr','contribution_margin_pct','dark_stores','cities','aov_inr','mtu_millions']]qc_latest.set_index('company').T

In [ ]:
# Chart 3: QC GOV trajectoryfig, ax = plt.subplots()for company, color, label in [('Eternal', ZOM, 'Blinkit'), ('Swiggy', SWG, 'Instamart')]:    d = qc[qc['company']==company]    ax.plot(d['quarter_end_date'], d['gov_inr_cr'], marker='o', linewidth=2.5, color=color, label=label)    ax.fill_between(d['quarter_end_date'], 0, d['gov_inr_cr'], color=color, alpha=0.08)ax.set_title('Quick commerce GOV by quarter (₹ crore)', fontsize=13, weight='bold', loc='left')ax.set_ylabel('GOV (₹ cr)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='upper left')plt.tight_layout()plt.show()

**Read:** Blinkit's absolute lead is wider than food delivery's. Blinkit GOV is 1.68× Instamart's. And unlike food delivery, this gap is *widening*, not stable.**QA-style edge case to flag:** Q3 FY26 shows Blinkit GOV actually *dipping* sequentially. Worth investigating — likely Q2 had a festive bump (Onam/Ganesh Chaturthi prep). Always read absolute and YoY together.

In [ ]:
# Chart 4: QC contribution margin — the centerpiece of Layer 2fig, ax = plt.subplots()for company, color, label in [('Eternal', ZOM, 'Blinkit'), ('Swiggy', SWG, 'Instamart')]:    d = qc[qc['company']==company]    ax.plot(d['quarter_end_date'], d['contribution_margin_pct'], marker='o', linewidth=2.5, color=color, label=label)ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)ax.text(qc['quarter_end_date'].min(), 0.2, 'breakeven', fontsize=9, alpha=0.7)ax.set_title('Quick commerce — Contribution margin (% of GOV)', fontsize=13, weight='bold', loc='left')ax.set_ylabel('Contribution margin (%)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='lower right')plt.tight_layout()plt.show()

In [ ]:
cm_pivot = qc.pivot(index='quarter_end_date', columns='company', values='contribution_margin_pct')cm_pivot['Gap (pp)'] = (cm_pivot['Eternal'] - cm_pivot['Swiggy']).round(2)cm_pivot = cm_pivot.rename(columns={'Eternal': 'Blinkit', 'Swiggy': 'Instamart'})cm_pivot

**Layer 2 finding (with a twist):**The popular narrative is "Blinkit is winning, Instamart is dying." The data is more nuanced.- Instamart's improvement: from −5.8% (Q1 FY25) to −1.8% (Q4 FY26) = **+4.0pp over 8 quarters**- Blinkit's improvement: from +2.4% (Q1 FY25) to +3.5% (Q4 FY26) = **+1.1pp over 8 quarters**Instamart is improving unit economics **3.6× faster than Blinkit** — just from below the waterline. The gap is narrowing, not widening.This doesn't mean Swiggy is winning. It means the company is making real operational progress that the headline narrative isn't crediting them for.**QA edge case:** This trend isn't monotone. Instamart's margin briefly worsened between Q2 and Q4 FY25 (from −6.4% → −4.2%, with adj. EBITDA spike to −18% due to dark store expansion). Always check whether the metric you're looking at is being optimized — or sacrificed.

In [ ]:
# Chart 5: Dark store productivityqc['gov_per_store_cr'] = (qc['gov_inr_cr'] / qc['dark_stores']).round(2)fig, ax = plt.subplots()for company, color, label in [('Eternal', ZOM, 'Blinkit'), ('Swiggy', SWG, 'Instamart')]:    d = qc[qc['company']==company]    ax.plot(d['quarter_end_date'], d['gov_per_store_cr'], marker='o', linewidth=2.5, color=color, label=label)ax.set_title('GOV per dark store per quarter (₹ crore)', fontsize=13, weight='bold', loc='left')ax.set_ylabel('GOV / store (₹ cr)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False)plt.tight_layout()plt.show()

**Read:** Two opposite trends crossing. Blinkit's per-store output is *declining* (₹7.7 → ₹7.0 cr) — the cost of expanding into less-dense neighborhoods. Instamart's productivity is *rising* (₹4.9 → ₹6.9 cr) — they've stopped opening new stores and let existing ones mature.Both are converging on ~₹7 cr/store/quarter. This looks like a physical ceiling for current-format dark stores in Indian Tier 1 cities. Whoever cracks it first opens the next leg of differentiation.

---## Synthesis: How does each parent allocate B2C GOV?This is where the layers come together. Both companies started in food delivery and added a QC business. Eight quarters later — where is each one's GOV actually coming from?

In [ ]:
# Combine food + QC datacombined = fd[['company','quarter','quarter_end_date','gov_inr_cr']].rename(columns={'gov_inr_cr':'food_gov'}).merge(    qc[['company','quarter','gov_inr_cr']].rename(columns={'gov_inr_cr':'qc_gov'}),    on=['company','quarter'])combined['total_b2c_gov'] = combined['food_gov'] + combined['qc_gov']combined['qc_share_pct'] = (combined['qc_gov'] / combined['total_b2c_gov'] * 100).round(1)combined.sort_values(['company','quarter_end_date'])

In [ ]:
# Chart 6: Stacked bar — business mix divergenceimport numpy as npfig, ax = plt.subplots(figsize=(11, 5.5))quarters = sorted(fd['quarter_end_date'].unique())x = np.arange(len(quarters))width = 0.35zom_fd = [fd[(fd['company']=='Eternal') & (fd['quarter_end_date']==q)]['gov_inr_cr'].iloc[0] for q in quarters]zom_qc = [qc[(qc['company']=='Eternal') & (qc['quarter_end_date']==q)]['gov_inr_cr'].iloc[0] for q in quarters]swg_fd = [fd[(fd['company']=='Swiggy') & (fd['quarter_end_date']==q)]['gov_inr_cr'].iloc[0] for q in quarters]swg_qc = [qc[(qc['company']=='Swiggy') & (qc['quarter_end_date']==q)]['gov_inr_cr'].iloc[0] for q in quarters]ax.bar(x - width/2, zom_fd, width, label='Zomato (food)', color=ZOM, alpha=0.55)ax.bar(x - width/2, zom_qc, width, bottom=zom_fd, label='Blinkit (QC)', color=ZOM)ax.bar(x + width/2, swg_fd, width, label='Swiggy (food)', color=SWG, alpha=0.55)ax.bar(x + width/2, swg_qc, width, bottom=swg_fd, label='Instamart (QC)', color=SWG)ax.set_xticks(x)ax.set_xticklabels([pd.to_datetime(q).strftime('%b %y') for q in quarters], rotation=0)ax.set_title('Total B2C GOV by company — food delivery + quick commerce', fontsize=13, weight='bold', loc='left')ax.set_ylabel('GOV (₹ cr)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='upper left', ncol=2)plt.tight_layout()plt.show()

In [ ]:
# Chart 7: QC share of total B2C GOVfig, ax = plt.subplots()zom_share = [qcv / (fdv + qcv) * 100 for fdv, qcv in zip(zom_fd, zom_qc)]swg_share = [qcv / (fdv + qcv) * 100 for fdv, qcv in zip(swg_fd, swg_qc)]ax.plot(quarters, zom_share, marker='o', linewidth=2.5, color=ZOM, label='Eternal: Blinkit share')ax.plot(quarters, swg_share, marker='o', linewidth=2.5, color=SWG, label='Swiggy: Instamart share')ax.axhline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.4)ax.text(quarters[0], 51, '50% — QC matches food delivery', fontsize=9, alpha=0.7)ax.set_title('Quick commerce share of total B2C GOV', fontsize=13, weight='bold', loc='left')ax.set_ylabel('QC / (QC + FD) (%)')ax.grid(axis='y', alpha=0.3)ax.legend(frameon=False, loc='lower right')plt.tight_layout()plt.show()

**The synthesis finding:**In Q1 FY25, both companies had similar mix: food delivery dominant, QC the second bet. By Q4 FY26:- **Eternal: 52% Blinkit, 48% Zomato.** Quick commerce has *passed* food delivery.- **Swiggy: 47% Instamart, 53% Swiggy food.** Food delivery still leads.Two companies that started identical have ended up in different categories. **Eternal is now best described as a quick-commerce company with a profitable food delivery business attached.** Swiggy is still a food-delivery company with a struggling quick-commerce business attached.That single strategic decision explains everything else: why Eternal's stock rerated to new highs and Swiggy's is 44% below its peak; why Eternal renamed itself ("Eternal" is brand-neutral; "Zomato" was food-delivery-locked); why Eternal split into separate apps.

---## ConclusionsThree takeaways, in plain English:### 1. Food delivery is a converged commodityEBITDA margin gap closed from 2.3pp → 0.1pp in 8 quarters. Both companies grow at 22–23% YoY and earn ~3.3–3.4% margin. Neither can win the future by being better at food delivery.### 2. Quick commerce is where the divergence livesBlinkit has 1.68× Instamart's GOV, the only positive contribution margin in the category, and is widening the absolute gap. But Instamart is improving unit economics 3.6× faster, just from below the waterline. The race isn't margin — it's scale.### 3. Eternal made quick commerce its main business; Swiggy didn't52% of Eternal's B2C GOV is Blinkit. 47% of Swiggy's is Instamart. Same starting point, different end state. **That's the single strategic decision that defines this comparison.**---## What I'd build nextWith access to internal data:1. **Cohort retention curves** — does a Q1 FY25 user still order in Q4 FY26?2. **SKU-mix contribution split** — which QC categories drive margin improvement?3. **Tier 1 vs Tier 2 store P&L** — at what stage does a Tier 2 store turn contribution-positive?## Caveats- 8 quarters is a short series. A single bad festive season could break these lines.- Disclosed metrics aren't perfectly like-for-like across companies. I've used the most consistent series available.- Estimates flagged in `data/SOURCES.md`. Where I had to triangulate, I noted it.- Three-player QC market not fully modeled (Zepto, Flipkart Minutes, BigBasket Now excluded).